# Computational Carpentry Project

Team members (replace placeholders before submission):
- Gregorio Adriano Rolli — Part A: Data structures and functions
- [Teammate 2 full name] — Part B: Stoichiometry and reaction balancing
- [Teammate 3 full name] — Part C: Simulation / Modeling

Parts B and C are reserved for the other two team members; adjust their assignments if needed.


## Part A — Data structures and functions

### 1. Load the periodic table


In [2]:
import pandas as pd

periodic_table = pd.read_csv("periodic_table.csv")
periodic_table.head()


,AtomicNumber,Symbol,Name,AtomicMass,CPKHexColor,ElectronConfiguration,Electronegativity,AtomicRadius,IonizationEnergy,ElectronAffinity,OxidationStates,StandardState,MeltingPoint,BoilingPoint,Density,GroupBlock,YearDiscovered
0,1,H,Hydrogen,1.008000,FFFFFF,1s1,2.20,120.0,13.598,0.754,"+1, -1",Gas,13.81,20.28,0.000090,Nonmetal,1766
1,2,He,Helium,4.002600,D9FFFF,1s2,NaN,140.0,24.587,NaN,0,Gas,0.95,4.22,0.000179,Noble gas,1868
2,3,Li,Lithium,7.000000,CC80FF,[He]2s1,0.98,182.0,5.392,0.618,+1,Solid,453.65,1615.00,0.534000,Alkali metal,1817
3,4,Be,Beryllium,9.012183,C2FF00,[He]2s2,1.57,153.0,9.323,NaN,+2,Solid,1560.00,2744.00,1.850000,Alkaline earth metal,1798
4,5,B,Boron,10.810000,FFB5B5,[He]2s2 2p1,2.04,192.0,8.298,0.277,+3,Solid,2348.00,4273.00,2.370000,Metalloid,1808


### 2. Dictionary

`.set_index()` replaces integer indexes by the `Symbol`. The key `"Symbol"` associates each symbol with its value `"AtomicMass"`; `to_dict()` converts that mapping into a Python dictionary.


In [3]:
atomic_masses = periodic_table.set_index("Symbol")["AtomicMass"].to_dict()

atomic_masses


{'H': 1.008,
 'He': 4.0026,
 'Li': 7.0,
 'Be': 9.012183,
 'B': 10.81,
 'C': 12.011,
 'N': 14.007,
 'O': 15.999,
 'F': 18.99840316,
 'Ne': 20.18,
 'Na': 22.9897693,
 'Mg': 24.305,
 'Al': 26.981538,
 'Si': 28.085,
 'P': 30.973762,
 'S': 32.07,
 'Cl': 35.45,
 'Ar': 39.9,
 'K': 39.0983,
 'Ca': 40.08,
 'Sc': 44.95591,
 'Ti': 47.867,
 'V': 50.9415,
 'Cr': 51.996,
 'Mn': 54.93804,
 'Fe': 55.84,
 'Co': 58.93319,
 'Ni': 58.693,
 'Cu': 63.55,
 'Zn': 65.4,
 'Ga': 69.723,
 'Ge': 72.63,
 'As': 74.92159,
 'Se': 78.97,
 'Br': 79.9,
 'Kr': 83.8,
 'Rb': 85.468,
 'Sr': 87.62,
 'Y': 88.90584,
 'Zr': 91.22,
 'Nb': 92.90637,
 'Mo': 95.95,
 'Tc': 96.90636,
 'Ru': 101.1,
 'Rh': 102.9055,
 'Pd': 106.42,
 'Ag': 107.868,
 'Cd': 112.41,
 'In': 114.818,
 'Sn': 118.71,
 'Sb': 121.76,
 'Te': 127.6,
 'I': 126.9045,
 'Xe': 131.29,
 'Cs': 132.905452,
 'Ba': 137.33,
 'La': 138.9055,
 'Ce': 140.116,
 'Pr': 140.90766,
 'Nd': 144.24,
 'Pm': 144.91276,
 'Sm': 150.4,
 'Eu': 151.964,
 'Gd': 157.2,
 'Tb': 158.92535,
 'Dy': 16

### 3. Molecular mass formula

Read the formula from left to right. Each element symbol begins with an uppercase letter and can include a lowercase letter. The digits following it give the number of atoms. Without digits, the count is 1. Multiply each count by the atomic mass from `atomic_masses` and add the contributions. Using `while` loops instead of `for in` give more control over each step.


In [ ]:
def molecular_mass(chemical_formula) -> float:
    """Returns the molecular mass of a chemical based on its formula"""
    if not isinstance(chemical_formula, str) or not chemical_formula:
        raise ValueError("The formula must be a non-empty string.")

    total_mass = 0.0
    i = 0

    while i < len(chemical_formula):
        # Read the element symbol, including an optional lowercase letter.
        if chemical_formula[i] not in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
            raise ValueError(f"Expected an element symbol at position {i + 1}.")
        symbol = chemical_formula[i]
        i += 1
        if i < len(chemical_formula) and chemical_formula[i] in "abcdefghijklmnopqrstuvwxyz":
            symbol += chemical_formula[i]
            i += 1

        if symbol not in atomic_masses:
            raise ValueError(f"Unknown element: {symbol}")

        # Read all digits so counts such as 12 are handled together.
        count_start = i
        while i < len(chemical_formula) and chemical_formula[i] in "0123456789":
            i += 1
        count_text = chemical_formula[count_start:i]
        count = int(count_text) if count_text else 1
        if count < 1:
            raise ValueError("Atom counts must be positive integers.")

        total_mass += atomic_masses[symbol] * count

    return total_mass


In [9]:
# Examples
for formula in ["H2O", "C6H12O6", "NaCl"]:
    print(f"{formula}: {molecular_mass(formula)}")


H2O: 18.015
C6H12O6: 180.156
NaCl: 58.4397693


### 4. Handle parentheses using recursion

When `parse_group` encounters `(`, it calls itself to calculate the mass inside the parentheses. That call returns at the matching `)`, then we multiply the group mass by the following count (or 1 if no number). This also handles nested parentheses.

Each call returns both its mass and the next unread position, so the outer call knows where to continue.


In [ ]:
def molecular_mass(chemical_formula):
    """Returns the molecular mass of a chemical based on its formula, including groups in parentheses."""
    if not isinstance(chemical_formula, str) or not chemical_formula:
        raise ValueError("The formula must be a non-empty string.")

    def read_count(i):
        """Read an optional positive integer and return the next position."""
        start = i
        while i < len(chemical_formula) and chemical_formula[i] in "0123456789":
            i += 1
        count = int(chemical_formula[start:i]) if i > start else 1
        if count < 1:
            raise ValueError("Atom and group counts must be positive integers.")
        return count, i

    def parse_group(i, inside_parentheses=False):
        total_mass = 0.0
        has_terms = False

        while i < len(chemical_formula):
            character = chemical_formula[i]

            if character == ")":
                if not inside_parentheses:
                    raise ValueError("Unexpected closing parenthesis.")
                if not has_terms:
                    raise ValueError("Parentheses cannot be empty.")
                return total_mass, i + 1

            if character == "(":
                # Recursively calculate the group, stopping at its matching ')'.
                term_mass, i = parse_group(i + 1, inside_parentheses=True)
            else:
                if character not in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
                    raise ValueError(f"Expected an element symbol at position {i + 1}.")
                symbol = character
                i += 1
                if i < len(chemical_formula) and chemical_formula[i] in "abcdefghijklmnopqrstuvwxyz":
                    symbol += chemical_formula[i]
                    i += 1
                if symbol not in atomic_masses:
                    raise ValueError(f"Unknown element: {symbol}")
                term_mass = atomic_masses[symbol]

            # The count applies to either one element or a whole group.
            count, i = read_count(i)
            total_mass += term_mass * count
            has_terms = True

        if inside_parentheses:
            raise ValueError("Missing closing parenthesis.")
        return total_mass, i

    total_mass, _ = parse_group(0)
    return total_mass


In [10]:
# Simple groups, multiple groups, and nested parentheses.
for formula in ["Ca(OH)2", "Al2(SO4)3", "(NH4)2SO4", "K4(ON(SO3)2)2"]:
    print(f"{formula}: {molecular_mass(formula):.3f} u")


Ca(OH)2: 74.094 u
Al2(SO4)3: 342.161 u
(NH4)2SO4: 132.144 u
K4(ON(SO3)2)2: 536.673 u


### 5. Handle hydration water

Split the formula at each dot (`.` or `·`). A leading number multiplies the entire component after it; if omitted, the coefficient is 1. For `CuSO4.5H2O`, add the mass of CuSO4 to five times the mass of H2O.

The updated `molecular_mass` keeps the recursive parser from point 4 as a local helper, `component_mass`, so each component can still contain parentheses. Points 3 and 4 remain above to show the progression. Run the notebook from top to bottom; teammates can then use `molecular_mass` for Part B. Results are in u (numerically equal to molar masses in g/mol).


In [11]:
def molecular_mass(chemical_formula):
    """Return molecular mass in u, including parentheses and hydration water."""
    if not isinstance(chemical_formula, str) or not chemical_formula:
        raise ValueError("The formula must be a non-empty string.")

    # Reuse the point 4 algorithm to calculate each component's mass.
    def component_mass(chemical_formula):
        """Return molecular mass in u, including groups in parentheses."""
        if not isinstance(chemical_formula, str) or not chemical_formula:
            raise ValueError("The formula must be a non-empty string.")

        def read_count(i):
            """Read an optional positive integer and return the next position."""
            start = i
            while i < len(chemical_formula) and chemical_formula[i] in "0123456789":
                i += 1
            count = int(chemical_formula[start:i]) if i > start else 1
            if count < 1:
                raise ValueError("Atom and group counts must be positive integers.")
            return count, i

        def parse_group(i, inside_parentheses=False):
            total_mass = 0.0
            has_terms = False

            while i < len(chemical_formula):
                character = chemical_formula[i]

                if character == ")":
                    if not inside_parentheses:
                        raise ValueError("Unexpected closing parenthesis.")
                    if not has_terms:
                        raise ValueError("Parentheses cannot be empty.")
                    return total_mass, i + 1

                if character == "(":
                    # Recursively calculate the group, stopping at its matching ')'.
                    term_mass, i = parse_group(i + 1, inside_parentheses=True)
                else:
                    if character not in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
                        raise ValueError(f"Expected an element symbol at position {i + 1}.")
                    symbol = character
                    i += 1
                    if i < len(chemical_formula) and chemical_formula[i] in "abcdefghijklmnopqrstuvwxyz":
                        symbol += chemical_formula[i]
                        i += 1
                    if symbol not in atomic_masses:
                        raise ValueError(f"Unknown element: {symbol}")
                    term_mass = atomic_masses[symbol]

                # The count applies to either one element or a whole group.
                count, i = read_count(i)
                total_mass += term_mass * count
                has_terms = True

            if inside_parentheses:
                raise ValueError("Missing closing parenthesis.")
            return total_mass, i

        total_mass, _ = parse_group(0)
        return total_mass

    total_mass = 0.0
    # Accept both a regular dot and the middle dot used in hydrate notation.
    components = chemical_formula.replace("·", ".").split(".")
    for component in components:
        if not component:
            raise ValueError("Each dot must separate non-empty formula components.")

        # A leading number multiplies the entire component: 5H2O = five waters.
        i = 0
        while i < len(component) and component[i] in "0123456789":
            i += 1
        coefficient = int(component[:i]) if i else 1
        if coefficient < 1:
            raise ValueError("Component coefficients must be positive integers.")
        formula = component[i:]
        if not formula:
            raise ValueError("A coefficient must be followed by a formula.")

        total_mass += coefficient * component_mass(formula)

    return total_mass


In [12]:
for formula in ["CuSO4.5H2O", "CuSO4·5H2O", "Na2CO3.10H2O", "Al2(SO4)3.18H2O"]:
    print(f"{formula}: {molecular_mass(formula):.3f} u")


CuSO4.5H2O: 249.691 u
CuSO4·5H2O: 249.691 u
Na2CO3.10H2O: 286.138 u
Al2(SO4)3.18H2O: 666.431 u
